# CS2 (국내반) --- Lecture 12: Graph Shooter (NumPy, Matplotlib)
### Problem Set (in-class)

*이 노트북은 채점되지 않습니다. 수업 시간에 바로 이어서 풀면 됩니다.*

- 함수 그래프를 쏴서 장애물(회색 원)을 피해 과녁(빨간 원)을 맞히는 게임
- 그래프는 발사대(검은 네모) 높이에서 출발. 장애물에 닿거나 화면 밖(|y| > 10)이면 **BLOCKED**, 과녁에 닿으면 **HIT**, 끝까지 가면 **MISS**
- 위에서부터 **순서대로** 실행: 앞에서 만든 함수(무기, 충돌 판정, 엔진)를 뒤 문제에서 그대로 씀
- 빈칸 `___`을 채워서 완성. 1~5번 필수, 6번 Optional

### Setup

아래 셀을 **먼저 한 번 실행**하세요. 게임의 레벨, 대결 맵, 그리기 함수가 들어 있는 `L13_game.py`를 만듭니다. (무기와 엔진은 여러분이 직접 만듭니다.)

In [ ]:
%%writefile L13_game.py
# L13_game.py : levels, battle maps and drawing for Graph Shooter.
# The functions YOU write (relu, insideCircle, shoot, ...) are not here.
import numpy as np
import matplotlib.pyplot as plt

np.seterr(all="ignore")       # no warnings for huge values such as np.exp(100)

# cannon: height of the cannon at x = -10
# walls:  obstacles (cx, cy, r)
# target: (tx, ty, r)
LEVELS = {
    1: dict(name="Straight Shot", cannon=0, walls=[], target=(8, 6, 0.7)),
    2: dict(name="Over the Hill", cannon=0, walls=[(0, -2, 4)], target=(8, 0, 0.7)),
    3: dict(name="Zigzag", cannon=0, walls=[(-4, 3, 2.5), (3, -3, 2.5)], target=(9, 2, 0.7)),
    4: dict(name="The Wall", cannon=-6,
            walls=[(0, -8, 2), (0, -4, 2), (0, 0, 2), (0, 4, 2)], target=(6, -6, 0.7)),
    5: dict(name="Step Up", cannon=-6,
            walls=[(-5, 1, 3.5), (5, -2, 3.5)], target=(8, 5, 0.7)),
}

def drawLevel(level, ax=None):
    if ax is None:
        plt.figure(figsize=(6, 6))
        ax = plt.gca()
    for cx, cy, r in level["walls"]:
        ax.add_patch(plt.Circle((cx, cy), r, color="gray"))
    tx, ty, tr = level["target"]
    ax.add_patch(plt.Circle((tx, ty), tr, color="red"))
    ax.plot(-10, level["cannon"], "ks", markersize=12)
    ax.set_xlim(-10, 10)
    ax.set_ylim(-10, 10)
    ax.set_aspect("equal")
    ax.grid(True)
    ax.set_title(level["name"])

def drawShot(level, result, xs, ys):
    """Draw one level and one shot: drawShot(level, *shoot(f, level))"""
    drawLevel(level)
    plt.plot(xs, ys, "b", linewidth=2)
    plt.title(level["name"] + ": " + result)
    plt.show()

def drawLevels(shots):
    """shots[i] = shoot(f, LEVELS[i + 1]). Draws every level with its shot."""
    plt.figure(figsize=(12, 8))
    for i in range(len(shots)):
        result, xs, ys = shots[i]
        ax = plt.subplot(2, 3, i + 1)
        drawLevel(LEVELS[i + 1], ax)
        ax.plot(xs, ys, "b", linewidth=2)
        ax.set_title("Level " + str(i + 1) + ": " + result)
    plt.tight_layout()
    plt.show()

# ---------------- battle ----------------
def battleMap(seed):
    """A random map: player A at x = -10, player B at x = 10, walls in the middle."""
    rng = np.random.default_rng(seed)
    walls = []
    for i in range(4):
        walls.append((rng.uniform(-5, 5), rng.uniform(-7, 7), rng.uniform(1, 2.5)))
    a, b = rng.uniform(-6, 6, 2)
    return dict(seed=seed, walls=walls, a=a, b=b)

def sideLevel(m, side):
    """The map seen by one player: both players shoot from the left (x = -10)."""
    if side == "A":
        return dict(name="A", cannon=m["a"], walls=m["walls"], target=(9.3, m["b"], 0.8))
    walls = [(-cx, cy, r) for cx, cy, r in m["walls"]]          # flip left and right
    return dict(name="B", cannon=m["b"], walls=walls, target=(9.3, m["a"], 0.8))

def drawBattle(m, shotA=None, shotB=None):
    plt.figure(figsize=(7, 7))
    ax = plt.gca()
    for cx, cy, r in m["walls"]:
        ax.add_patch(plt.Circle((cx, cy), r, color="gray"))
    ax.plot(-10, m["a"], "s", color="C0", markersize=14)
    ax.plot(10, m["b"], "s", color="C1", markersize=14)
    ax.text(-9.4, m["a"] + 0.9, "A", color="C0", fontsize=14, weight="bold")
    ax.text(8.9, m["b"] + 0.9, "B", color="C1", fontsize=14, weight="bold")
    title = "Battle map " + str(m["seed"])
    if shotA is not None:
        ax.plot(shotA[1], shotA[2], color="C0", linewidth=2)
        ax.plot(-shotB[1], shotB[2], color="C1", linewidth=2)          # B flies right to left
        title = "A: " + shotA[0] + "   B: " + shotB[0]
    ax.set_xlim(-10.5, 10.5)
    ax.set_ylim(-10, 10)
    ax.set_aspect("equal")
    ax.grid(True)
    ax.set_title(title)
    plt.show()

def battle(playerA, playerB, seed, shoot):
    """Both players fire at the same time with YOUR shoot function."""
    m = battleMap(seed)
    shotA = shoot(playerA, sideLevel(m, "A"))
    shotB = shoot(playerB, sideLevel(m, "B"))
    drawBattle(m, shotA, shotB)
    if shotA[0] == "HIT" and shotB[0] == "HIT":
        return "Draw! Both hit."
    if shotA[0] == "HIT":
        return "A wins!"
    if shotB[0] == "HIT":
        return "B wins!"
    return "No winner. Change your functions and fire again!"

def drawCatalog(functions):
    """functions: a dictionary {name: function}. Draws each one."""
    x = np.linspace(-4, 4, 801)
    plt.figure(figsize=(12, 6))
    k = 0
    for name in functions:
        k = k + 1
        plt.subplot(2, 3, k)
        plt.axhline(0, color="0.7")
        plt.axvline(0, color="0.7")
        plt.plot(x, functions[name](x), linewidth=2)
        plt.xlim(-4, 4)
        plt.ylim(-3, 4)
        plt.title(name)
    plt.tight_layout()
    plt.show()

### Problem 1

(Weapons from Deep Learning)

1. Write functions `relu`, `leakyRelu`, `sigmoid`, `tanh`:
   - input parameter: an array `x`
   - return value: `x`의 각 원소에 대한 함수값의 array
     - ReLU: $x$와 $0$ 중 큰 값 (`np.maximum`)
     - Leaky ReLU: $x > 0$이면 $x$, 아니면 $0.1x$ (`np.where`)
     - sigmoid: $\dfrac{1}{1 + e^{-x}}$
     - tanh: $\dfrac{e^{x} - e^{-x}}{e^{x} + e^{-x}}$ (`np.tanh`를 쓰지 말고 공식으로)
2. Write a function `drawFunctions`:
   - action: $-5 \le x \le 5$에서 네 함수의 그래프를 그림. 보이는 범위는 x: $-5 \sim 5$, y: $-2 \sim 5$ (빈칸 채우기)

*$e^{x}$는 `np.exp(x)`. 여기서 만든 함수가 Problem 4, 5의 **무기**가 됩니다.*

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def relu(x):
    return ___

def leakyRelu(x):
    return ___

def sigmoid(x):
    return ___

def tanh(x):
    return ___

def drawFunctions():
    x = np.linspace(___, ___, 201)
    plt.plot(x, relu(x), label="ReLU")
    plt.plot(x, leakyRelu(x), "--", label="Leaky ReLU")
    plt.plot(x, sigmoid(x), label="sigmoid")
    plt.plot(x, tanh(x), label="tanh")
    plt.xlim(___, ___)
    plt.ylim(___, ___)
    plt.grid(True)
    plt.legend()
    plt.show()

p = np.array([-2, 0, 3])
print(relu(p))                  # [0 0 3]
print(leakyRelu(p))             # [-0.2  0.   3. ]
print(np.round(sigmoid(p), 3))  # [0.119 0.5   0.953]
print(np.round(tanh(p), 3))     # [-0.964  0.     0.995]
drawFunctions()

### Problem 2

(Collision)

1. Write a function `insideCircle`:
   - input parameters: arrays `x`, `y`, numbers `cx`, `cy`, `r`
   - return value: 점 `(x[i], y[i])`가 중심 `(cx, cy)`, 반지름 `r`인 원 안에 있으면 True인 mask
2. Write a function `firstTrue`:
   - input parameter: a mask `mask`
   - return value: 처음으로 True가 나오는 위치. True가 하나도 없으면 `len(mask)`
3. Write a function `drawPoints`:
   - action: 원 안의 점은 빨간색, 원 밖의 점은 파란색으로 그림 (빈칸 채우기)

*`np.argmax(mask)`는 처음 True의 위치. 단, True가 없으면 0을 주므로 `np.any`로 먼저 확인. `inside == False`는 원 밖인 자리가 True인 mask.*

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def insideCircle(x, y, cx, cy, r):
    return ___

def firstTrue(mask):
    if ___:
        return ___
    return len(mask)

def drawPoints(px, py, inside):
    plt.plot(___, ___, "r.")
    plt.plot(___, ___, "b.")
    plt.gca().set_aspect("equal")
    plt.show()

x = np.array([0, 1, 2, 3, 4])
y = np.array([0, 0, 0, 0, 0])
inside = insideCircle(x, y, 3, 0, 1.5)
print(inside)                                           # [False False  True  True  True]
print(np.any(insideCircle(x, y, 10, 0, 1)))             # False
print(firstTrue(np.array([False, False, True, True])))  # 2
print(firstTrue(np.array([False, False, False])))       # 3

rng = np.random.default_rng(13)      # 500 random points
px = rng.uniform(-5, 5, 500)
py = rng.uniform(-5, 5, 500)
drawPoints(px, py, insideCircle(px, py, 1, 1, 3))

### Problem 3

(The Shooting Engine) Complete the function `shoot` (빈칸 2개):
- input parameters: a function `f`, a level (dictionary: `cannon`, `walls`, `target`)
- return value: 결과(`"HIT"`, `"BLOCKED"`, `"MISS"`)와 총알이 지나간 `x`, `y`
- 빈칸 1: 그래프를 옮겨서 발사대 높이(`level["cannon"]`)에서 출발
- 빈칸 2: 장애물 `(cx, cy, r)` 안에 있는 점도 멈추는 곳 mask `stop`에 추가

*Problem 2의 `insideCircle`, `firstTrue`를 이용. `stop | mask`는 원소마다 or (둘 중 하나라도 True면 True).*

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from L13_game import drawShot

def shoot(f, level):
    x = np.linspace(-10, 10, 2001)
    y = f(x) + 0 * x                     # f may return just a number
    y = np.nan_to_num(y, nan=100, posinf=100, neginf=-100)  # inf: off
    y = ___                      # start at the cannon
    stop = np.abs(y) > 10                # off the screen
    for cx, cy, r in level["walls"]:
        stop = stop | ___
    end = firstTrue(stop)                # where the shot stops
    tx, ty, tr = level["target"]
    hit = insideCircle(x[:end], y[:end], tx, ty, tr)
    if np.any(hit):
        k = firstTrue(hit) + 1
        return "HIT", x[:k], y[:k]
    if end < len(x):
        return "BLOCKED", x[:end + 1], y[:end + 1]
    return "MISS", x, y

def flat(x):
    return 0 * x

def hill(x):
    return -(x + 10) * (x - 8) / 20

def down(x):
    return -0.3 * x

level = dict(name="Test", cannon=0, walls=[(0, 0, 2)], target=(8, 0, 0.7))
print(shoot(flat, level)[0])  # BLOCKED
print(shoot(hill, level)[0])  # HIT
print(shoot(down, level)[0])  # MISS

result, xs, ys = shoot(hill, level)
drawShot(level, result, xs, ys)

### Problem 4

(Mission: Clear the Levels) 레벨 1~5의 과녁을 맞히는 함수 `f1` ~ `f5`를 작성. 과녁에 맞기만 하면 어떤 함수든 정답.
- 무기: Problem 1에서 만든 `relu`, `leakyRelu`, `sigmoid`, `tanh`와 `np.sin`, `np.cos`, `np.abs`, ... 도우미 함수를 `def`로 더 만들어도 됨
- 엔진: Problem 3에서 만든 `shoot`. 출력이 모두 `HIT`이면 성공
- `def` 고치기 → 실행 → 결과 보고 다시 고치기. `LEVEL`을 1~5로 바꾸면 그 레벨만 크게 그림

| 식 | 모양 |
|---|---|
| `a * x` | 직선, `a`가 클수록 가파름 |
| `-(x + 10) * (x - 8)` | x = -10, x = 8을 지나는 산 모양 |
| `a * np.sin(b * x)` | 물결: `a`는 높이, `b`가 클수록 촘촘 |
| `a * relu(x - c)` | x = c까지 평평, 그다음 직선 |
| `a * sigmoid(b * (x - c))` | x = c 근처에서 계단 오르기 |
| `a * tanh(x - c)` | x = c 근처에서 -a에서 a로 |
| `f(x - c)` | 오른쪽으로 c만큼 이동 |

In [ ]:
import numpy as np
from L13_game import LEVELS, drawShot, drawLevels

# Weapons: relu, leakyRelu, sigmoid, tanh (P01), np.sin, np.cos, np.abs, ...
# Any function that hits the target is a correct answer!
def f1(x):
    return ___

def f2(x):
    return ___

def f3(x):
    return ___

def f4(x):
    return ___

def f5(x):
    return ___

print(shoot(f1, LEVELS[1])[0])  # HIT
print(shoot(f2, LEVELS[2])[0])  # HIT
print(shoot(f3, LEVELS[3])[0])  # HIT
print(shoot(f4, LEVELS[4])[0])  # HIT
print(shoot(f5, LEVELS[5])[0])  # HIT

fs = [f1, f2, f3, f4, f5]
shots = []
for i in range(5):
    shots.append(shoot(fs[i], LEVELS[i + 1]))
drawLevels(shots)

LEVEL = 0       # 1 ~ 5: draw only that level, bigger
if LEVEL > 0:
    result, xs, ys = shoot(fs[LEVEL - 1], LEVELS[LEVEL])
    drawShot(LEVELS[LEVEL], result, xs, ys)

### Problem 5

(Battle!) 짝꿍과 한 컴퓨터에서 대결. 같은 맵(`SEED`)에서 A는 왼쪽, B는 오른쪽에 있음.
- A는 `playerA`, B는 `playerB`를 작성 (자기 무기 선택). B도 왼쪽에서 쏘는 것처럼 작성 (좌우는 게임이 뒤집어 줌)
- `battle(playerA, playerB, SEED, shoot)`: 두 사람이 동시에 발사. 엔진은 Problem 3에서 만든 `shoot`
- 상대를 맞히면 승리. 둘 다 빗나가면 함수를 고쳐서 다시 발사. `SEED`를 바꾸면 새 맵

In [ ]:
import numpy as np
from L13_game import battleMap, drawBattle, battle

SEED = 4                    # pick a map together: 1, 2, 3, ...
drawBattle(battleMap(SEED))  # look at the map first

def playerA(x):             # A fires from the left
    return ___

def playerB(x):             # B also aims as if firing from the left
    return ___

print(battle(playerA, playerB, SEED, shoot))

### Optional Problem (Problem 6)

필수 문제와 달리 제출/검사 대상은 아니지만, 시간이 남으면 시도해보는 것을 권합니다.

(More Weapons) Write functions `step`, `elu`, `softplus`, `swish`, `myAbs`:
- step: $x > 0$이면 $1$, 아니면 $0$
- ELU: $x > 0$이면 $x$, 아니면 $e^{x} - 1$
- softplus: $\log(1 + e^{x})$ (`np.log`)
- swish: $x \times \text{sigmoid}(x)$ (Problem 1의 `sigmoid` 이용)
- myAbs: $|x|$ (`np.abs`를 쓰지 말고 `np.where`로)

*완성한 무기는 Problem 4, 5 셀을 다시 실행해서 바로 사용 가능.*

In [ ]:
import numpy as np
from L13_game import drawCatalog

def step(x):
    return ___

def elu(x):
    return ___

def softplus(x):
    return ___

def swish(x):
    return ___

def myAbs(x):
    return ___

p = np.array([-2, 0, 3])
print(step(p))                   # [0 0 1]
print(np.round(elu(p), 3))       # [-0.865  0.     3.   ]
print(np.round(softplus(p), 3))  # [0.127 0.693 3.049]
print(np.round(swish(p), 3))     # [-0.238  0.     2.858]
print(myAbs(p))                  # [2 0 3]
drawCatalog({"step": step, "elu": elu, "softplus": softplus,
             "swish": swish, "myAbs": myAbs, "sigmoid": sigmoid})